## HW 8

### Download and splitting dataset

In [26]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [27]:
# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Create DataFrame
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [28]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train_0 = X_train[y_train == 0]
X_train_1 = X_train[y_train == 1]
X_train_2 = X_train[y_train == 2]

print("Shape of X_train_0:", X_train_0.shape)
print("Shape of X_train_1:", X_train_1.shape)
print("Shape of X_train_2:", X_train_2.shape)

Shape of X_train_0: (40, 4)
Shape of X_train_1: (41, 4)
Shape of X_train_2: (39, 4)


### Priors, Means, Covariance Matrices

In [29]:
# Prior probs
prior_0 = len(X_train_0) / len(X_train)
prior_1 = len(X_train_1) / len(X_train)
prior_2 = len(X_train_2) / len(X_train)

print(f"Prior probability(0): {prior_0:.4f}")
print(f"Prior probability(1): {prior_1:.4f}")
print(f"Prior probability(2): {prior_2:.4f}")


#  Means
mu_0 = np.mean(X_train_0, axis=0)
mu_1 = np.mean(X_train_1, axis=0)
mu_2 = np.mean(X_train_2, axis=0)

print(f"Mean(0): {mu_0}")
print(f"Mean(1): {mu_1}")
print(f"Mean(2): {mu_2}")

# Covariance
cov_0 = np.cov(X_train_0.T)
cov_1 = np.cov(X_train_1.T)
cov_2 = np.cov(X_train_2.T)

print(f"Covariance(0): {cov_0}")
print(f"Covariance(1): {cov_1}")
print(f"Covariance(2): {cov_2}")

Prior probability(0): 0.3333
Prior probability(1): 0.3417
Prior probability(2): 0.3250
Mean(0): [4.99   3.4525 1.45   0.245 ]
Mean(1): [5.9195122  2.77073171 4.24146341 1.32195122]
Mean(2): [6.53333333 2.96666667 5.52051282 2.        ]
Covariance(0): [[0.12707692 0.10797436 0.01897436 0.0094359 ]
 [0.10797436 0.15640385 0.01371795 0.00808974]
 [0.01897436 0.01371795 0.03384615 0.00666667]
 [0.0094359  0.00808974 0.00666667 0.01125641]]
Covariance(1): [[0.29410976 0.10108537 0.19592073 0.06056098]
 [0.10108537 0.10262195 0.0962439  0.04715854]
 [0.19592073 0.0962439  0.2314878  0.08131707]
 [0.06056098 0.04715854 0.08131707 0.0422561 ]]
Covariance(2): [[0.42754386 0.10114035 0.31429825 0.04947368]
 [0.10114035 0.10175439 0.09280702 0.06026316]
 [0.31429825 0.09280702 0.29325236 0.05210526]
 [0.04947368 0.06026316 0.05210526 0.08421053]]


### QDA

In [30]:
from numpy.linalg import inv, det

def qda_score(x, mu, cov, prior):
    inv_cov = inv(cov) # inverse covariance matrix
    det_cov = det(cov) # covariance determinant
    diff = x - mu

    score = -0.5 * (diff.T @ inv_cov @ diff) - 0.5 * np.log(det_cov) + np.log(prior)
    return score

### Test

In [31]:
predictions = []

for x in X_test:
    # calc scores for all classes
    score_0 = qda_score(x, mu_0, cov_0, prior_0)
    score_1 = qda_score(x, mu_1, cov_1, prior_1)
    score_2 = qda_score(x, mu_2, cov_2, prior_2)
    
    # find the higest score
    best_class = np.argmax([score_0, score_1, score_2])
    predictions.append(best_class)

predictions = np.array(predictions)

# Accuracy results
accuracy = np.mean(predictions == y_test)
print(f"Accuracy manual QDA: {accuracy:.4f}")

Accuracy manual QDA: 0.9667


### `sklearn` QDA

In [32]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# Initialize the QDA model
qda_sklearn = QuadraticDiscriminantAnalysis()

# Train the model
qda_sklearn.fit(X_train, y_train)
sklearn_predictions = qda_sklearn.predict(X_test)

# Calculate the accuracy
sklearn_accuracy = qda_sklearn.score(X_test, y_test)

print(f"Accuracy sklearn QDA: {sklearn_accuracy:.4f}")

Accuracy sklearn QDA: 0.9667


In [33]:
# Custom QDA vs Sklearn vs Actual 
comparison_df = pd.DataFrame({
    'Custom': predictions,
    'Actual': y_test,
    'Sklearn': sklearn_predictions
})

print(comparison_df.head(15))

    Custom  Actual  Sklearn
0        1       1        1
1        0       0        0
2        2       2        2
3        1       1        1
4        1       1        1
5        0       0        0
6        1       1        1
7        2       2        2
8        2       1        2
9        1       1        1
10       2       2        2
11       0       0        0
12       0       0        0
13       0       0        0
14       0       0        0


### Conclusion:

The results demonstrate that the analytical calculations of the custom QDA model perfectly match the standard `scikit-learn` implementation. 

Both approaches returned the exact same accuracy(~96.67%) and identical predictions on the test dataset, confirming the mathematical correctness (calculating means, covariance matrices, and applying the discriminant function) of the manual algorithm.

In the comparison table, both the custom model and the `scikit-learn` model misclassified the record at index 8 in the exact same way.